<a href="https://github.com/gutris1/segsmaker">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

---
> **Segsmaker Repofusion (Colab)** - Repo2d baseline with Repo1a parallel downloader flow.
>
> Run the installer first, then use the form-based downloader and launcher cells below.


In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount__GDrive = 'No' # @param ["Yes", "No"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/gutris1/segsmaker/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)

## Model & LoRA Downloader
<span style="font-size:13px;">
Fill up to 5 checkpoint/model URLs and 5 LoRA URLs. Empty slots are skipped automatically.<br>
Supported sources: <code>civitai.com</code>, <code>huggingface.co</code>, GitHub raw links, direct URLs, and Google Drive.
</span>


In [ ]:
# @title Model & LoRA Downloader - Parallel Slots {"display-mode":"form"}
# @markdown ### Checkpoints / Models
# @markdown > Format: `URL` or `URL optional_filename.safetensors`
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### LoRA
# @markdown > Format: `URL` or `URL optional_filename.safetensors`
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### Optional VAE
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### Speed Options
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

import os
import re
import sys
import shlex
import threading
import subprocess
import requests
from pathlib import Path
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output

try:
    import nenen88 as _sm_downloader
except Exception as _import_error:
    raise RuntimeError('Run the WebUI Installer cell first so nenen88 and path variables are available.') from _import_error

_TO_BYTES = {
    'B': 1,
    'KiB': 1024,
    'MiB': 1024 ** 2,
    'GiB': 1024 ** 3,
    'KB': 1000,
    'MB': 1000 ** 2,
    'GB': 1000 ** 3,
}

def _parse_aria2_stats(raw):
    stats = {'pct': 0, 'done_b': 0.0, 'total_b': 0.0, 'speed_b': 0.0, 'eta_s': 0}
    match = re.search(r'([\d.]+)(\w+)/([\d.]+)(\w+)\((\d+)%\)', raw)
    if match:
        stats['done_b'] = float(match.group(1)) * _TO_BYTES.get(match.group(2), 1)
        stats['total_b'] = float(match.group(3)) * _TO_BYTES.get(match.group(4), 1)
        stats['pct'] = int(match.group(5))
    match = re.search(r'DL:([\d.]+)(\w+)', raw)
    if match:
        stats['speed_b'] = float(match.group(1)) * _TO_BYTES.get(match.group(2), 1)
    match = re.search(r'ETA:(\d+)(s|m|h)', raw)
    if match:
        stats['eta_s'] = int(match.group(1)) * {'s': 1, 'm': 60, 'h': 3600}[match.group(2)]
    return stats

def _fmt_size(byte_count):
    value = float(byte_count)
    for unit in ('B', 'KiB', 'MiB', 'GiB'):
        if value < 1024 or unit == 'GiB':
            return f'{value:.1f}{unit}'
        value /= 1024

def _fmt_eta(seconds):
    if seconds <= 0:
        return '?'
    return f'{seconds}s' if seconds < 60 else f'{seconds // 60}m{seconds % 60:02d}s'

def _fmt_progress(raw):
    return raw.replace('[', '?', 1).replace(']', '?', 1)

def _call_if_present(name, *args):
    func = getattr(_sm_downloader, name, None)
    if callable(func):
        return func(*args)
    return None

def _parallel_ariari(url, fp, fn, on_progress=None):
    resolved = _sm_downloader.get_url(url, fn)
    if not resolved or not resolved[0]:
        return

    resolved_url, metadata, version_id = resolved
    civitai_domain = _sm_downloader.get_civdom(resolved_url)
    token = getattr(_sm_downloader, 'TOKET', '')
    hf_token = getattr(_sm_downloader, 'TOBRUT', '')
    user_agent = _sm_downloader.civitai_headers().get('User-Agent', 'CivitaiLink:Automatic1111') if civitai_domain else 'Mozilla/5.0'

    if civitai_domain and f'{civitai_domain}/api/download/models/' in resolved_url and token:
        try:
            headers = {'User-Agent': user_agent, 'Authorization': f'Bearer {token}'}
            response = requests.get(resolved_url, headers=headers, allow_redirects=True, stream=True, timeout=30)
            final_url = response.url
            response.close()
            if final_url and final_url != resolved_url:
                resolved_url = final_url
        except Exception as error:
            print(f'Civitai preflight failed; falling back to aria2 headers: {error}')

    cmd = [
        'aria2c',
        f'--header=User-Agent: {user_agent}',
        '--allow-overwrite=true',
        '--console-log-level=error',
        '--stderr=true',
        '--auto-file-renaming=false',
        '--min-split-size=1M',
        f'--dir={str(fp)}',
        '-c',
        '-x16',
        '-s16',
        '-k1M',
        '-j5',
    ]

    if civitai_domain and f'{civitai_domain}/api/download/models/' in resolved_url and token:
        cmd.append(f'--header=Authorization: Bearer {token}')
    if hf_token and 'huggingface.co' in resolved_url:
        cmd.append(f'--header=Authorization: Bearer {hf_token}')
    if fn:
        cmd += ['-o', fn]
    cmd.append(resolved_url)

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    aria2_output = ''
    errors = []

    while True:
        line = process.stderr.readline()
        if line == '' and process.poll() is not None:
            break
        if not line:
            continue
        aria2_output += line
        for progress_line in line.splitlines():
            if 'errorCode' in progress_line or 'Exception' in progress_line:
                errors.append(progress_line)
            if re.match(r'\[#\w{6}\s.*\]', progress_line):
                if on_progress:
                    on_progress(progress_line)
                else:
                    print(f"\r{' ' * 300}\r {_fmt_progress(progress_line)}", end='')
                    sys.stdout.flush()
                break

    process.wait()
    if process.returncode != 0:
        detail = '\n'.join(errors) if errors else aria2_output[-1000:]
        raise RuntimeError(f'aria2c failed for {url}\n{detail}')

    if metadata:
        _call_if_present('civitai_infotags', metadata, fp, fn, version_id)
        _call_if_present('civitai_preview', metadata, fp, fn, version_id)

def _install_parallel_fallback():
    if hasattr(_sm_downloader, 'parallel_batch_download'):
        return _sm_downloader.parallel_batch_download

    def parallel_batch_download(items, max_workers=3):
        if not items:
            return {}

        total = len(items)
        results = {}
        state_lock = threading.Lock()
        state = {}
        stop_render = threading.Event()

        def set_state(index, **values):
            with state_lock:
                state.setdefault(index, {'label': '', 'raw': '', 'done': False, 'ok': True})
                state[index].update(values)

        def build_frame():
            with state_lock:
                snapshot = {key: dict(value) for key, value in state.items()}

            lines = []
            aggregate_speed = 0.0
            aggregate_done = 0.0
            aggregate_total = 0.0
            aggregate_eta = 0
            active = 0

            for index in range(total):
                slot = snapshot.get(index, {})
                label = slot.get('label', f'file {index + 1}')
                done = slot.get('done', False)
                ok = slot.get('ok', True)
                raw = slot.get('raw', '')

                if done:
                    icon = '?' if ok else '?'
                    lines.append(f'[{index + 1}/{total}] {icon} {label}')
                elif raw:
                    lines.append(_fmt_progress(raw))
                    stats = _parse_aria2_stats(raw)
                    aggregate_speed += stats['speed_b']
                    aggregate_done += stats['done_b']
                    aggregate_total += stats['total_b']
                    aggregate_eta = max(aggregate_eta, stats['eta_s'])
                    active += 1
                else:
                    lines.append(f'[{index + 1}/{total}] starting {label}')

            if active:
                pct = int(100 * aggregate_done / aggregate_total) if aggregate_total else 0
                lines.append(
                    f'?#Parallel {_fmt_size(aggregate_done)}/{_fmt_size(aggregate_total)}'
                    f'({pct}%) DL:{_fmt_size(aggregate_speed)}/s ETA:{_fmt_eta(aggregate_eta)}?'
                )
            else:
                completed = sum(1 for slot in snapshot.values() if slot.get('done'))
                lines.append(f'Parallel completed {completed}/{total}')

            return '\n'.join(lines)

        def render_loop():
            while not stop_render.is_set():
                clear_output(wait=True)
                print(build_frame())
                sys.stdout.flush()
                stop_render.wait(0.25)
            clear_output(wait=True)
            print(build_frame())
            sys.stdout.flush()

        def worker(index, url, destination, filename):
            label = filename or Path(urlparse(url).path).name or url
            set_state(index, label=label)
            try:
                target = Path(destination).expanduser()
                target.mkdir(parents=True, exist_ok=True)

                def on_progress(raw):
                    set_state(index, raw=raw)

                is_accelerated = any(domain in url for domain in [*_sm_downloader.CIVITAI, 'huggingface.co', 'github.com'])
                if is_accelerated:
                    _parallel_ariari(url, target, filename, on_progress=on_progress)
                elif 'drive.google.com' in url:
                    _sm_downloader.gdrown(url, target, filename)
                else:
                    command = f"curl -#OJL '{url}'" if not filename else f"curl -#L '{url}' -o '{filename}'"
                    old_cwd = Path.cwd()
                    os.chdir(target)
                    try:
                        _sm_downloader.curlly(command, filename or label)
                    finally:
                        os.chdir(old_cwd)

                set_state(index, done=True, ok=True, raw='')
                return url, 'ok'
            except Exception as error:
                set_state(index, done=True, ok=False, raw='', label=f'{label} ({error})')
                return url, 'error'

        renderer = threading.Thread(target=render_loop, daemon=True, name='repofusion-progress-renderer')
        renderer.start()
        try:
            with ThreadPoolExecutor(max_workers=max(1, int(max_workers))) as executor:
                futures = [
                    executor.submit(worker, index, url, destination, filename)
                    for index, (url, destination, filename) in enumerate(items)
                ]
                for future in as_completed(futures):
                    url, status = future.result()
                    results[url] = status
        finally:
            stop_render.set()
            renderer.join(timeout=2)

        ok_count = sum(1 for status in results.values() if status == 'ok')
        error_count = sum(1 for status in results.values() if status == 'error')
        print(f'Batch complete - {ok_count} ok, {error_count} error(s)')
        return results

    _sm_downloader.parallel_batch_download = parallel_batch_download
    return parallel_batch_download

parallel_batch_download = _install_parallel_fallback()

def _parse_slot(raw_value, destination):
    raw_value = raw_value.strip()
    if not raw_value:
        return None
    parts = raw_value.split(None, 1)
    filename = parts[1].strip() if len(parts) > 1 else None
    return (parts[0].strip(), str(destination), filename)

_queue = []
for _raw_url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    item = _parse_slot(_raw_url, CKPT)
    if item:
        _queue.append(item)
for _raw_url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    item = _parse_slot(_raw_url, LORA)
    if item:
        _queue.append(item)
if VAE_URL.strip():
    item = _parse_slot(VAE_URL, VAE)
    if item:
        _queue.append(item)

if not _queue:
    print('No URLs provided - skipping downloads.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    for _url, _destination, _filename in _queue:
        get_ipython().run_line_magic('cd', f'-q {_destination}')
        _line = _url if not _filename else f'{_url} {_filename}'
        get_ipython().run_line_magic('download', _line)


## ControlNet *(Optional)*


In [ ]:
# @title ControlNet Widget
%run $Controlnet_Widget


## Launch WebUI
<span style="font-size:13px;">
Pick a launch profile instead of copying command-line arguments manually. Use Custom when you want full manual control.
</span>


In [ ]:
# @title Launch WebUI {"display-mode":"form"}
Launch_Profile = 'Auto from selected WebUI' # @param ["Auto from selected WebUI", "A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI", "Custom"]
Extra_Args = '' # @param {type:"string", placeholder:"optional extra args, tunnel token, or overrides"}
Skip_Widget = False # @param {type:"boolean"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}

_default_args = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

_selected_profile = Webui if Launch_Profile == 'Auto from selected WebUI' else Launch_Profile
_arg_parts = []
if _selected_profile != 'Custom':
    _arg_parts.append(_default_args.get(_selected_profile, ''))
if Extra_Args.strip():
    _arg_parts.append(Extra_Args.strip())
if Skip_Widget:
    _arg_parts.append('--skip-widget')
if Skip_ComfyUI_Check:
    _arg_parts.append('--skip-comfyui-check')

_args = ' '.join(part for part in _arg_parts if part).strip()
print(f'Launching {_selected_profile} with args: {_args or "<none>"}')
get_ipython().run_line_magic('cd', f'-q {WebUI}')
get_ipython().run_line_magic('run', f'segsmaker.py {_args}'.strip())
